# Rally E2B RP Merged Upload

Uploads the completed Rally E2B A100/B75 merged checkpoint from the browser export kernel output to Hugging Face without downloading the checkpoint locally.

In [ ]:
import os, platform, shutil
from pathlib import Path

print('python_platform=', platform.platform())
print('working_disk_free_gb=', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))
print('input_dirs=', [str(p) for p in Path('/kaggle/input').glob('*')])


In [ ]:
import os, subprocess, sys, time

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    '--upgrade',
    'huggingface_hub[cli]>=1.5.0',
    'hf_transfer>=0.1.9',
])

secret_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN') or ''
for attempt in range(20):
    if secret_token:
        break
    try:
        from kaggle_secrets import UserSecretsClient
        secret_token = UserSecretsClient().get_secret('HF_TOKEN')
        break
    except Exception as exc:
        print('hf_secret_attempt_failed=', attempt + 1, type(exc).__name__)
        time.sleep(10)
if secret_token and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = secret_token
if secret_token and not os.environ.get('HUGGING_FACE_HUB_TOKEN'):
    os.environ['HUGGING_FACE_HUB_TOKEN'] = secret_token
print('hf_secret_loaded=', bool(secret_token))
if not os.environ.get('HF_TOKEN'):
    raise RuntimeError('HF_TOKEN Kaggle secret is required for merged checkpoint upload')


In [ ]:
import json, os
from pathlib import Path
from huggingface_hub import HfApi

input_root = Path(os.environ.get('RALLY_EXPORT_INPUT_ROOT', '/kaggle/input/rally-e2b-browser-export'))
candidates = [
    input_root / 'rally-e2b-browser-export' / 'a100-b75-merged',
    input_root / 'a100-b75-merged',
    input_root / 'notebooks' / 'rally-e2b-browser-export' / 'a100-b75-merged',
    Path('/kaggle/input/notebooks/rally-e2b-browser-export/a100-b75-merged'),
]
merged_dir = next((path for path in candidates if (path / 'model.safetensors').exists()), None)
if merged_dir is None:
    checked = '\n'.join(str(path) for path in candidates)
    raise FileNotFoundError(f'Could not find a100-b75 merged checkpoint. Checked:\n{checked}')

repo_id = os.environ.get('RALLY_RP_MERGED_REPO', 'thomasjvu/rally-2b-rp-a100-b75-merged')
private = os.environ.get('RALLY_PRIVATE', '1') != '0'
api = HfApi(token=os.environ['HF_TOKEN'])
api.create_repo(repo_id=repo_id, repo_type='model', private=private, exist_ok=True)
commit = api.upload_folder(
    repo_id=repo_id,
    repo_type='model',
    folder_path=str(merged_dir),
    commit_message='Upload Rally E2B two-stage RP A100/B75 merged checkpoint from Kaggle',
)
report = {
    'ok': True,
    'repo_id': repo_id,
    'private': private,
    'source_dir': str(merged_dir),
    'commit_url': str(commit),
    'files': sorted(path.name for path in merged_dir.iterdir() if path.is_file()),
}
report_path = Path('/kaggle/working/rally-e2b-rp-merged-upload-report.json')
report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + '\n')
print(json.dumps(report, indent=2, sort_keys=True))
